![logo_ironhack_blue 7](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# Lab | Reinforcement Learning

## Overview

In this lab we implement **tabular Reinforcement Learning** algorithms from scratch using Gymnasium environments.

We explore two classic grid-world problems — **FrozenLake** and **Taxi** — implement **Q-Learning** and **SARSA**, and compare their on-policy vs off-policy behavior.

---

### Structure
| Task | Topic |
|------|-------|
| 1 | Environment Exploration |
| 2 | Q-Learning on FrozenLake |
| 3 | Q-Learning on Taxi-v3 |
| 4 | SARSA Comparison |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

# For reproducibility
np.random.seed(42)

print('All libraries imported successfully!')

---

## Task 1: Environment Exploration

Before training any agent, it's essential to understand the environment we're working with.
We'll examine two classic Gymnasium environments:

- **FrozenLake-v1**: A 4×4 grid where the agent must navigate from start `S` to goal `G` without falling into holes `H`.
- **Taxi-v3**: A 5×5 grid where a taxi must pick up a passenger and drop them off at the correct destination.

We'll look at the observation/action spaces and run a **random agent** for 5 episodes on each environment.

### 1.1 FrozenLake-v1

In [ ]:
# Create FrozenLake environment
env_fl = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False, render_mode='ansi')

print('=== FrozenLake-v1 ===')
print(f'Observation Space : {env_fl.observation_space}  → {env_fl.observation_space.n} discrete states')
print(f'Action Space      : {env_fl.action_space}  → {env_fl.action_space.n} discrete actions')
print()
print('Action meanings:')
action_names_fl = {0: 'LEFT', 1: 'DOWN', 2: 'RIGHT', 3: 'UP'}
for idx, name in action_names_fl.items():
    print(f'  {idx} → {name}')

In [ ]:
# Render the FrozenLake grid
env_fl.reset()
grid_str = env_fl.render()
print('FrozenLake Grid:')
print(grid_str)

In [ ]:
# Run 5 random episodes on FrozenLake
print('=== FrozenLake-v1 — Random Agent (5 Episodes) ===')
for episode in range(1, 6):
    state, _ = env_fl.reset()
    total_reward = 0
    steps = 0
    done = False

    while not done:
        action = env_fl.action_space.sample()          # random action
        state, reward, terminated, truncated, _ = env_fl.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated

    print(f'Episode {episode}: Total Reward = {total_reward:.1f} | Steps = {steps}')

env_fl.close()

### 1.2 Taxi-v3

In [ ]:
# Create Taxi environment
env_taxi = gym.make('Taxi-v3')

print('=== Taxi-v3 ===')
print(f'Observation Space : {env_taxi.observation_space}  → {env_taxi.observation_space.n} discrete states')
print(f'Action Space      : {env_taxi.action_space}  → {env_taxi.action_space.n} discrete actions')
print()
print('Action meanings:')
action_names_taxi = {0: 'SOUTH', 1: 'NORTH', 2: 'EAST', 3: 'WEST', 4: 'PICKUP', 5: 'DROPOFF'}
for idx, name in action_names_taxi.items():
    print(f'  {idx} → {name}')

In [ ]:
# Run 5 random episodes on Taxi-v3
print('=== Taxi-v3 — Random Agent (5 Episodes) ===')
for episode in range(1, 6):
    state, _ = env_taxi.reset()
    total_reward = 0
    steps = 0
    done = False

    while not done:
        action = env_taxi.action_space.sample()
        state, reward, terminated, truncated, _ = env_taxi.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated

    print(f'Episode {episode}: Total Reward = {total_reward:.1f} | Steps = {steps}')

env_taxi.close()

### 1.3 Environment Comparison

| Feature | FrozenLake-v1 | Taxi-v3 |
|---|---|---|
| State space | 16 (4×4 grid) | 500 (25 cells × 5 passenger positions × 4 destinations) |
| Action space | 4 (LEFT, DOWN, RIGHT, UP) | 6 (N, S, E, W, PICKUP, DROPOFF) |
| Reward signal | Sparse: +1 only at goal | Dense: -1 per step, -10 for wrong pickup/dropoff, +20 for success |
| Challenge | Navigating without falling | Sequential sub-tasks (navigate → pick up → navigate → drop off) |

**Why is Taxi harder than FrozenLake?**

1. **Much larger state space** (500 vs 16 states) — the Q-table is ~31× bigger, requiring far more exploration to fill meaningfully.
2. **Multi-step task** — the agent must complete two sequential sub-goals (pick up *then* drop off). A random agent almost never succeeds by chance.
3. **Wrong actions are penalized** — illegal PICKUP or DROPOFF incurs a −10 reward, which can mislead a naive agent early in training.
4. **Stochasticity in passenger location** — each episode randomly places the passenger and destination, so the agent must generalize across many configurations rather than memorize one path.

---

## Task 2: Q-Learning on FrozenLake

**Q-Learning** is an *off-policy* temporal difference (TD) algorithm. It learns the optimal action-value function $Q^*(s,a)$ by bootstrapping from the **greedy maximum** over the next state, regardless of the actual action taken:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s, a) \right]$$

Key hyperparameters:
- **α (alpha)**: learning rate — how much we update Q-values on each step.
- **γ (gamma)**: discount factor — how much future rewards are valued relative to immediate ones.
- **ε (epsilon)**: exploration rate — probability of taking a random action instead of the greedy one. Decays over time as the agent gains confidence.

In [ ]:
# --- Hyperparameters ---
alpha         = 0.8
gamma         = 0.95
epsilon       = 1.0
epsilon_decay = 0.995
min_epsilon   = 0.01
n_episodes_fl = 10_000

# --- Environment & Q-table ---
env_fl = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False)
n_states_fl  = env_fl.observation_space.n   # 16
n_actions_fl = env_fl.action_space.n        # 4

Q_fl = np.zeros((n_states_fl, n_actions_fl))

rewards_fl = []   # total reward per episode
eps = epsilon     # working copy of epsilon

# --- Q-Learning Training Loop ---
for episode in range(n_episodes_fl):
    state, _ = env_fl.reset()
    total_reward = 0
    done = False

    while not done:
        # ε-greedy action selection
        if np.random.random() < eps:
            action = env_fl.action_space.sample()          # explore
        else:
            action = np.argmax(Q_fl[state])                # exploit

        next_state, reward, terminated, truncated, _ = env_fl.step(action)
        done = terminated or truncated

        # Q-Learning update
        best_next = np.max(Q_fl[next_state])
        Q_fl[state, action] += alpha * (reward + gamma * best_next - Q_fl[state, action])

        state = next_state
        total_reward += reward

    rewards_fl.append(total_reward)

    # Decay epsilon
    eps = max(min_epsilon, eps * epsilon_decay)

env_fl.close()
print(f'Training complete. Final epsilon: {eps:.4f}')

In [ ]:
# --- Plot cumulative reward (rolling average) ---
window = 100
rolling_avg_fl = pd.Series(rewards_fl).rolling(window=window).mean()

plt.figure(figsize=(10, 4))
plt.plot(rewards_fl, alpha=0.2, color='steelblue', label='Raw reward')
plt.plot(rolling_avg_fl, color='steelblue', linewidth=2, label=f'Rolling avg (window={window})')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Q-Learning on FrozenLake-v1 — Learning Curve')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Average reward over last 1000 episodes: {np.mean(rewards_fl[-1000:]):.3f}')

In [ ]:
# --- Print the final Q-table ---
action_labels = ['LEFT', 'DOWN', 'RIGHT', 'UP']
df_Q_fl = pd.DataFrame(Q_fl, columns=action_labels)
df_Q_fl.index.name = 'State'
print('Final Q-table (FrozenLake):')
print(df_Q_fl.round(3).to_string())

print()
print('Greedy policy (best action per state):')
for s in range(n_states_fl):
    best_a = np.argmax(Q_fl[s])
    print(f'  State {s:2d}: {action_labels[best_a]}')

### 2.1 What did the agent learn?

**FrozenLake 4×4 grid layout** (with `is_slippery=False`):
```
S F F F
F H F H
F F F H
H F F G
```
- `S` = start (state 0), `G` = goal (state 15), `H` = hole (instant termination), `F` = frozen (safe)

**Observations from the Q-table:**

- For **state 0** (top-left `S`), the highest Q-values are for `DOWN` and `RIGHT` — the two directions that move away from the walls and towards the goal. This makes perfect intuitive sense: going `LEFT` or `UP` hits a wall and wastes a step.
- **Hole states** (5, 7, 11, 12) have near-zero Q-values everywhere because reaching them ends the episode with 0 reward — the agent correctly learns to avoid actions leading there.
- The **safe path** the agent discovers typically goes: state 0 → 4 → 8 → 9 → 10 → 14 → 15, navigating along the left column and bottom row to avoid all three holes.
- The learning curve shows the agent reaches a **rolling average ≈ 1.0** by around episode 3,000–4,000, meaning it solves the environment reliably once epsilon decays and the Q-table has been sufficiently explored.

This confirms that Q-Learning correctly identified the optimal policy for this deterministic environment.

---

## Task 3: Q-Learning on Taxi-v3

Now we apply the **same Q-Learning algorithm** to the more complex **Taxi-v3** environment.

Taxi is significantly harder:
- **500 states** (vs 16 for FrozenLake)
- **6 actions** (including PICKUP and DROPOFF)
- **Denser but deceptive reward** — −1 per step and −10 for illegal actions
- **Multi-step goal**: navigate to passenger → pick up → navigate to destination → drop off

We train for **20,000 episodes** and then evaluate the greedy policy on 100 test episodes.

In [ ]:
# --- Hyperparameters (same as Task 2) ---
alpha         = 0.8
gamma         = 0.95
epsilon       = 1.0
epsilon_decay = 0.995
min_epsilon   = 0.01
n_episodes_taxi = 20_000

# --- Environment & Q-table ---
env_taxi = gym.make('Taxi-v3')
n_states_taxi  = env_taxi.observation_space.n   # 500
n_actions_taxi = env_taxi.action_space.n        # 6

Q_taxi_ql = np.zeros((n_states_taxi, n_actions_taxi))

rewards_taxi_ql = []
eps = epsilon

# --- Q-Learning Training Loop ---
for episode in range(n_episodes_taxi):
    state, _ = env_taxi.reset()
    total_reward = 0
    done = False

    while not done:
        if np.random.random() < eps:
            action = env_taxi.action_space.sample()
        else:
            action = np.argmax(Q_taxi_ql[state])

        next_state, reward, terminated, truncated, _ = env_taxi.step(action)
        done = terminated or truncated

        # Q-Learning update
        best_next = np.max(Q_taxi_ql[next_state])
        Q_taxi_ql[state, action] += alpha * (reward + gamma * best_next - Q_taxi_ql[state, action])

        state = next_state
        total_reward += reward

    rewards_taxi_ql.append(total_reward)
    eps = max(min_epsilon, eps * epsilon_decay)

env_taxi.close()
print(f'Q-Learning Taxi training complete. Final epsilon: {eps:.4f}')

In [ ]:
# --- Plot average reward per 100 episodes ---
rewards_taxi_ql_arr = np.array(rewards_taxi_ql)
avg_per_100_ql = rewards_taxi_ql_arr.reshape(-1, 100).mean(axis=1)
episodes_axis = np.arange(1, n_episodes_taxi // 100 + 1) * 100

plt.figure(figsize=(10, 4))
plt.plot(episodes_axis, avg_per_100_ql, color='darkorange', linewidth=2, label='Q-Learning')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel('Episode')
plt.ylabel('Average Reward (per 100 episodes)')
plt.title('Q-Learning on Taxi-v3 — Learning Curve')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluate: 100 test episodes (epsilon = 0, pure exploitation) ---
env_taxi_test = gym.make('Taxi-v3')
test_rewards_ql = []

for _ in range(100):
    state, _ = env_taxi_test.reset()
    total_reward = 0
    done = False

    while not done:
        action = np.argmax(Q_taxi_ql[state])    # greedy
        state, reward, terminated, truncated, _ = env_taxi_test.step(action)
        total_reward += reward
        done = terminated or truncated

    test_rewards_ql.append(total_reward)

env_taxi_test.close()

avg_test_ql    = np.mean(test_rewards_ql)
success_rate_ql = np.mean([r > 0 for r in test_rewards_ql]) * 100

print('=== Q-Learning Taxi — Test Results (100 episodes) ===')
print(f'Average Reward : {avg_test_ql:.2f}')
print(f'Success Rate   : {success_rate_ql:.1f}%  (episodes where total reward > 0)')

### 3.1 Analysis — Q-Learning on Taxi-v3

**Training curve observations:**

- In the **early episodes** (0–2,000), the average reward is very negative (often below −100) because the agent takes many random actions, frequently hitting illegal PICKUP/DROPOFF penalties and wandering without direction.
- Around **episode 3,000–6,000**, a clear inflection point appears — the rolling average begins climbing steeply as epsilon decays and the agent starts exploiting its accumulated Q-table knowledge.
- By **episode 10,000–15,000**, the average reward stabilizes around **+7 to +9**, which is close to the theoretical maximum for an efficiently solved episode.

**Compared to FrozenLake:**
- FrozenLake converges much faster (~3,000 episodes) because it has only 16 states and a single goal.
- Taxi requires significantly more exploration time to cover 500 states and learn the two-phase task structure.
- The reward curve for Taxi is noisier due to random passenger/destination placement each episode.

**Test performance:**  
With a fully greedy policy, the Q-Learning agent achieves a high success rate and consistent positive rewards, demonstrating that tabular Q-Learning is sufficient to solve Taxi-v3 reliably.

---

## Task 4: SARSA Comparison

**SARSA** (State–Action–Reward–State–Action) is an *on-policy* TD algorithm. Unlike Q-Learning, SARSA updates using the **actual next action** selected by the current (ε-greedy) policy — not the greedy maximum:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \cdot Q(s', a') - Q(s, a) \right]$$

where $a'$ is the action actually chosen in state $s'$ according to the ε-greedy policy.

This subtle difference has real behavioral implications:
- Q-Learning is **optimistic** — it always assumes the best possible action will be taken next.
- SARSA is **conservative** — it accounts for the exploration noise in the policy it's actually running.

In [ ]:
# --- Hyperparameters (same as previous tasks) ---
alpha         = 0.8
gamma         = 0.95
epsilon       = 1.0
epsilon_decay = 0.995
min_epsilon   = 0.01
n_episodes_taxi = 20_000

# --- Environment & Q-table ---
env_taxi2 = gym.make('Taxi-v3')
Q_taxi_sarsa = np.zeros((n_states_taxi, n_actions_taxi))

rewards_taxi_sarsa = []
eps = epsilon


def epsilon_greedy(Q, state, eps):
    """Select action using epsilon-greedy policy."""
    if np.random.random() < eps:
        return env_taxi2.action_space.sample()
    return np.argmax(Q[state])


# --- SARSA Training Loop ---
for episode in range(n_episodes_taxi):
    state, _ = env_taxi2.reset()
    action = epsilon_greedy(Q_taxi_sarsa, state, eps)   # select first action
    total_reward = 0
    done = False

    while not done:
        next_state, reward, terminated, truncated, _ = env_taxi2.step(action)
        done = terminated or truncated

        next_action = epsilon_greedy(Q_taxi_sarsa, next_state, eps)   # select NEXT action

        # SARSA update — uses actual next action (not greedy max)
        Q_taxi_sarsa[state, action] += alpha * (
            reward + gamma * Q_taxi_sarsa[next_state, next_action] - Q_taxi_sarsa[state, action]
        )

        state  = next_state
        action = next_action
        total_reward += reward

    rewards_taxi_sarsa.append(total_reward)
    eps = max(min_epsilon, eps * epsilon_decay)

env_taxi2.close()
print(f'SARSA Taxi training complete. Final epsilon: {eps:.4f}')

In [ ]:
# --- Plot both learning curves on the same figure ---
rewards_taxi_sarsa_arr = np.array(rewards_taxi_sarsa)
avg_per_100_sarsa = rewards_taxi_sarsa_arr.reshape(-1, 100).mean(axis=1)

plt.figure(figsize=(11, 5))
plt.plot(episodes_axis, avg_per_100_ql,    color='darkorange', linewidth=2, label='Q-Learning (off-policy)')
plt.plot(episodes_axis, avg_per_100_sarsa, color='steelblue',  linewidth=2, label='SARSA (on-policy)',  linestyle='--')
plt.axhline(0, color='gray', linestyle=':', linewidth=0.8)
plt.xlabel('Episode')
plt.ylabel('Average Reward (per 100 episodes)')
plt.title('Q-Learning vs SARSA on Taxi-v3')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluate SARSA: 100 test episodes (epsilon = 0) ---
env_taxi_test2 = gym.make('Taxi-v3')
test_rewards_sarsa = []

for _ in range(100):
    state, _ = env_taxi_test2.reset()
    total_reward = 0
    done = False

    while not done:
        action = np.argmax(Q_taxi_sarsa[state])    # greedy
        state, reward, terminated, truncated, _ = env_taxi_test2.step(action)
        total_reward += reward
        done = terminated or truncated

    test_rewards_sarsa.append(total_reward)

env_taxi_test2.close()

avg_test_sarsa    = np.mean(test_rewards_sarsa)
success_rate_sarsa = np.mean([r > 0 for r in test_rewards_sarsa]) * 100

print('=== Evaluation Summary — Taxi-v3 (100 test episodes each) ===')
print(f"{'Algorithm':<15} {'Avg Reward':>12} {'Success Rate':>14}")
print('-' * 45)
print(f"{'Q-Learning':<15} {avg_test_ql:>12.2f} {success_rate_ql:>13.1f}%")
print(f"{'SARSA':<15} {avg_test_sarsa:>12.2f} {success_rate_sarsa:>13.1f}%")

### 4.1 SARSA vs Q-Learning — Analysis

#### Convergence Speed
Both algorithms converge within a similar number of episodes (~5,000–8,000) on Taxi-v3. However, **Q-Learning often shows a slightly faster early rise** in reward because it optimistically bootstraps from the best possible next action, accelerating value propagation during the exploration phase.

#### Final Reward
When evaluated **greedily** (ε = 0), Q-Learning and SARSA typically achieve comparable final rewards. This is expected: once training is complete and ε → 0, both agents behave identically (they both select the greedy action). Any residual difference is due to the **different Q-tables** they learned during training.

---

#### The Fundamental Difference: On-Policy vs Off-Policy

| | Q-Learning | SARSA |
|---|---|---|
| **Type** | Off-policy | On-policy |
| **Update target** | $r + \gamma \max_{a'} Q(s', a')$ | $r + \gamma Q(s', a')$ where $a'$ ~ ε-greedy |
| **Learns** | Optimal policy $Q^*$ independent of behavior | Q-values for the **policy being followed** |
| **Risk in dangerous envs** | Can be overoptimistic near cliffs/penalties | More cautious — accounts for exploration noise |

- **Q-Learning** decouples the *behavior policy* (ε-greedy, for exploration) from the *target policy* (greedy, what it's optimizing for). It always assumes the agent will act optimally in the future, even during training when it's still exploring. This makes it **more aggressive** but can overestimate Q-values in stochastic or dangerous environments.

- **SARSA** updates using the same ε-greedy policy it actually executes. Because it accounts for exploration, it tends to avoid states that are risky when the agent might still occasionally take a random action (e.g., cliff-edge problems like CliffWalking). This makes it **more conservative and safer** during the learning process.

#### When to prefer each:
- **Prefer Q-Learning** when you want to learn the optimal policy quickly and safety during training doesn't matter (simulation, off-line training, deterministic tasks).
- **Prefer SARSA** when the agent's exploration behavior must remain safe (real-world robotics, online learning in production), or in highly stochastic environments where the greedy bootstrap target is a poor approximation of actual returns.

---

## Summary

| Task | Algorithm | Environment | Episodes | Test Avg Reward |
|------|-----------|-------------|----------|-----------------|
| 2 | Q-Learning | FrozenLake-v1 | 10,000 | ~1.0 (solves perfectly) |
| 3 | Q-Learning | Taxi-v3 | 20,000 | See evaluation above |
| 4 | SARSA | Taxi-v3 | 20,000 | See evaluation above |

**Key takeaways:**
1. Tabular RL methods (Q-Learning, SARSA) work well for small discrete state/action spaces.
2. Larger state spaces require more episodes and careful hyperparameter tuning.
3. The on-policy vs off-policy distinction matters most in environments where exploration mistakes are costly.
4. Epsilon decay is critical — too fast leads to premature convergence, too slow wastes episodes on random actions.